# ICU-SEWS · Colab GRU 학습
런타임 → 런타임 유형 변경 → **GPU(T4 등)** 를 선택하고 위에서부터 실행하세요.

- Google Drive의 기존 전처리 파일 또는 `training.zip`을 사용합니다.
- 모델 학습은 별도 **Python 3.10** 환경에서 실행됩니다. 노트북 자체 Python 버전은 바꾸지 않습니다.
- 공식 dev 85% / test 15%와 환자 5-fold를 그대로 사용합니다.
- 기본 실행은 **fold 0, 최대 30 epochs**, 검증 AUPRC로 조기 종료합니다.
- 체크포인트는 매 epoch Drive에 저장합니다. 재연결 후 같은 셀을 실행하면 마지막 완료 epoch부터 재개합니다.
- Test 평가, 확률 보정, 경보 임계값 선택은 이 노트북에서 하지 않습니다. 출력 점수는 보정 전 개발 결과입니다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
# 데이터 담당의 파일이 있는 위치. 필요한 경우 이 경로만 수정하세요.
SOURCE_ROOT = Path("/content/drive/MyDrive/capston")
# 첫 실행에서 생성한 GRU 입력과 학습 결과를 보관합니다.
WORK_ROOT = Path("/content/drive/MyDrive/SEWS_GRU")
RUN_NAME = "gru_v1"
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print("원본/전처리 파일:", SOURCE_ROOT)
print("결과 저장:", WORK_ROOT / "results" / RUN_NAME)

## 1. 코드와 Python 3.10 GPU 환경 준비
초기 설치에 시간이 걸릴 수 있습니다. 런타임이 초기화되면 다시 실행하세요. GPU가 배정되지 않으면 학습을 시작하지 않습니다.

In [ ]:
import subprocess
import sys
import shutil

REPO_URL = "https://github.com/sogang-icu-sews/ICU-SEWS.git"
BRANCH = "codex/gru-colab"
REPO = Path("/content/ICU-SEWS")
def run(args, **kwargs):
    return subprocess.run([str(x) for x in args], check=True, **kwargs)

if not REPO.exists():
    run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO])
else:
    origin = subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip()
    assert origin == REPO_URL, "다른 저장소가 같은 경로에 있습니다."
    # 현재 코드를 유지합니다. 재개 시 코드가 달라지면 학습기가 중단합니다.
print(subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True))
run(["nvidia-smi"])
run([sys.executable, "-m", "pip", "install", "-q", "uv"])
UV = [sys.executable, "-m", "uv"]
ENV = Path("/content/sews-py310")
PY = ENV / "bin/python"
if not PY.exists():
    run(UV + ["venv", "--python", "3.10", str(ENV)])
run(UV + ["pip", "install", "--python", str(PY), "torch==2.8.0",
          "--index-url", "https://download.pytorch.org/whl/cu126"])
run(UV + ["pip", "install", "--python", str(PY), "-r", str(REPO / "requirements-gru-colab.txt")])
run([PY, "-c", "import sys,torch; print(sys.version); print(torch.__version__); "
     "assert sys.version_info[:2] == (3,10); assert torch.cuda.is_available(), 'GPU 런타임을 선택하세요'; "
     "print(torch.cuda.get_device_name(0))"])


## 2. 학습 입력 준비
**기존 전처리 파일**이 있으면 재사용합니다. 필요한 것은:
- `data/seq_values.npy`, `seq_row_start.npy`, `eval_meta.parquet`, `features.parquet` (또는 `gru_static.parquet`)
- `artifacts/preprocess_config.json`, `feature_registry.json`, `seq_meta.json`, `patient_split.csv`
- `artifacts/preprocess_params/fold0.json` ~ `fold4.json`

없으면 `SOURCE_ROOT/training.zip`을 준비하세요. ZIP 안에 `training_setA`, `training_setB` 폴더와 PSV 파일이 있어야 합니다. 원본에서 필요한 입력만 만들며 로지스틱 회귀는 실행하지 않습니다.

큰 입력은 Colab 임시 디스크에 복사해서 읽고, 결과는 Drive에 저장합니다. 준비가 중간에 실패하면 새 런타임에서 다시 실행하세요.

In [ ]:
import json

LOCAL_ROOT = Path("/content/sews_gru_inputs")
CACHED_ROOT = WORK_ROOT / "prepared"
data_files = ["seq_values.npy", "seq_row_start.npy", "eval_meta.parquet"]
art_files = ["preprocess_config.json", "feature_registry.json", "seq_meta.json", "patient_split.csv"]
param_files = [f"preprocess_params/fold{k}.json" for k in range(5)]

def ready(root):
    static_ok = any((root / "data" / p).is_file() for p in ["gru_static.parquet", "features.parquet"])
    return static_ok and all((root / "data" / p).is_file() for p in data_files) and all(
        (root / "artifacts" / p).is_file() for p in art_files + param_files)

def copy_inputs(source, destination):
    static = "gru_static.parquet" if (source / "data/gru_static.parquet").exists() else "features.parquet"
    files = [("data", p) for p in data_files + [static]]
    files += [("artifacts", p) for p in art_files + param_files]
    marker = source / "artifacts/gru_preparation_complete.json"
    if marker.exists():
        files.append(("artifacts", "gru_preparation_complete.json"))
    provenance = source / "artifacts/gru_source_files.csv"
    if provenance.exists():
        files.append(("artifacts", "gru_source_files.csv"))
    for folder, name in files:
        target = destination / folder / name
        target.parent.mkdir(parents=True, exist_ok=True)
        temp = target.with_name(target.name + ".partial")
        shutil.copyfile(source / folder / name, temp)
        temp.replace(target)

if not ready(LOCAL_ROOT):
    prepared = next((p for p in [CACHED_ROOT, SOURCE_ROOT] if ready(p)), None)
    if prepared is not None:
        copy_inputs(prepared, LOCAL_ROOT)
    else:
        source_zip = SOURCE_ROOT / "training.zip"
        assert source_zip.is_file(), f"전처리 파일 또는 {source_zip}이 필요합니다."
        local_zip = Path("/content/sews_training.zip")
        shutil.copyfile(source_zip, local_zip)
        run([PY, REPO / "scripts/prepare_gru_colab.py", "--archive", local_zip, "--out", LOCAL_ROOT])
        copy_inputs(LOCAL_ROOT, CACHED_ROOT)
assert ready(LOCAL_ROOT)
print("학습 입력 준비 완료:", LOCAL_ROOT)


## 3. GRU 학습
처음에는 `FOLDS = [0]`으로 실행하세요. 결과 확인 후 `[0, 1, 2, 3, 4]`로 변경하고 다시 실행하면 fold 0 체크포인트를 재사용하며 나머지 fold를 학습합니다.

- `POS_WEIGHT=True`: 학습 fold의 음성/양성 행 비율로 가중치를 계산합니다.
- 하이퍼파라미터를 바꾸는 실험은 **RUN_NAME도 새 이름으로 변경**하세요.
- 같은 실행에서 `EPOCHS`를 늘릴 수 있습니다. 조기 종료된 fold는 재개해도 추가 학습하지 않습니다.
- 매 epoch 진행 상황이 표시됩니다. 한 epoch 도중 끊기면 그 epoch는 처음부터 다시 실행합니다.
- 재현성 확인은 CPU에서 수행했습니다. 실제 Colab GPU 성능과 실행 시간은 첫 실행에서 확인해야 합니다.

In [ ]:
FOLDS = [0]          # 확인 후 [0, 1, 2, 3, 4]
EPOCHS = 30
BATCH_SIZE = 128
HIDDEN_SIZE = 64
LAYERS = 1
DROPOUT = 0.2
LEARNING_RATE = 0.001
POS_WEIGHT = True
OUT = WORK_ROOT / "results" / RUN_NAME

command = [PY, "-u", REPO / "scripts/train_gru.py",
           "--root", LOCAL_ROOT, "--out", OUT, "--device", "cuda",
           "--folds", *map(str, FOLDS), "--epochs", str(EPOCHS),
           "--batch-size", str(BATCH_SIZE), "--hidden-size", str(HIDDEN_SIZE),
           "--layers", str(LAYERS), "--dropout", str(DROPOUT),
           "--learning-rate", str(LEARNING_RATE), "--patience", "5", "--resume"]
if POS_WEIGHT:
    command.append("--pos-weight")
run(command)


## 4. 결과 확인
- `fold*/best.pt`: 검증 AUPRC가 가장 높았던 모델 및 전처리 기준
- `fold*/latest.pt`: 중단 후 재개용 상태
- `fold*/history.json`: epoch별 손실·검증 AUPRC·소요 시간
- `fold*/oof.parquet`: 해당 검증 fold의 예측
- `summary.json`, `oof_predictions.parquet`: 이번에 선택한 fold들의 통합 결과

`complete_5fold=false`면 아직 전체 5-fold 결과가 아닙니다. 경보 기준이나 임상 위험 확률로 바로 사용하지 마세요. 다른 모델과 비교할 때 같은 환자 분할·평가 구간을 사용합니다.

In [ ]:
summary = json.loads((OUT / "summary.json").read_text())
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("Google Drive 결과 위치:", OUT)
